# 04 - Build data source, index, indexer, and run keyword query


## Step 1 - Load environment and create Search clients


In [ ]:
import os

from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient, SearchIndexerClient
from dotenv import load_dotenv

load_dotenv("../env/.env")
subscription_id = os.getenv("AZURE_SUBSCRIPTION_ID")
resource_group = os.getenv("AZURE_RESOURCE_GROUP")
search_endpoint = os.getenv("SEARCH_ENDPOINT")
storage_account = os.getenv("STORAGE_ACCOUNT_NAME")
container_name = os.getenv("STORAGE_CONTAINER_NAME", "nasa")
if not all([subscription_id, resource_group, search_endpoint, storage_account]):
    raise RuntimeError("Missing required settings in env/.env")
credential = DefaultAzureCredential()
index_client = SearchIndexClient(endpoint=search_endpoint, credential=credential)
indexer_client = SearchIndexerClient(endpoint=search_endpoint, credential=credential)
index_name = "nasa-index"
data_source_name = "nasa-datasource"
indexer_name = "nasa-indexer"


## Step 2 - Create data source using ResourceId connection string


In [ ]:
from azure.search.documents.indexes.models import SearchIndexerDataContainer, SearchIndexerDataSourceConnection

# ResourceId=... connection string (no key) tells Search to authenticate to
# Storage using the Search service's system-assigned managed identity. The
# corresponding Storage Blob Data Reader role assignment is defined in
# infra/main.bicep on search.identity.principalId. No `identity=` kwarg is
# needed for the system-assigned MI path — that parameter is only used to
# specify a user-assigned managed identity via SearchIndexerDataUserAssignedIdentity.
connection_string = (
    f"ResourceId=/subscriptions/{subscription_id}/resourceGroups/{resource_group}"
    f"/providers/Microsoft.Storage/storageAccounts/{storage_account};"
)
data_source = SearchIndexerDataSourceConnection(
    name=data_source_name,
    type="azureblob",
    connection_string=connection_string,
    container=SearchIndexerDataContainer(name=container_name),
)
indexer_client.create_or_update_data_source_connection(data_source)
print(f"Data source ready: {data_source_name}")


## Step 3 - Create index


In [ ]:
from azure.search.documents.indexes.models import SearchField, SearchFieldDataType, SearchIndex, SimpleField

fields = [
    SimpleField(name="metadata_storage_path", type=SearchFieldDataType.String, key=True),
    SearchField(name="title", type=SearchFieldDataType.String, searchable=True),
    SearchField(name="content", type=SearchFieldDataType.String, searchable=True),
]
index = SearchIndex(name=index_name, fields=fields)
index_client.create_or_update_index(index)
print(f"Index ready: {index_name}")


## Step 4 - Create indexer and poll status


In [ ]:
import time

from azure.search.documents.indexes.models import FieldMapping, FieldMappingFunction, SearchIndexer

indexer = SearchIndexer(
    name=indexer_name,
    data_source_name=data_source_name,
    target_index_name=index_name,
    field_mappings=[
        FieldMapping(source_field_name="metadata_storage_name", target_field_name="title"),
        FieldMapping(
            source_field_name="metadata_storage_path",
            target_field_name="metadata_storage_path",
            mapping_function=FieldMappingFunction(name="base64Encode"),
        ),
    ],
)
indexer_client.create_or_update_indexer(indexer)

# Capture the previous run's end_time BEFORE triggering the new run so we can
# distinguish stale results from the current execution. Without this, a re-run
# of this cell would see the prior run's last_result.status == "success" and
# exit immediately, skipping the actual new run.
previous_end_time = None
try:
    previous_status = indexer_client.get_indexer_status(indexer_name)
    if previous_status.last_result:
        previous_end_time = previous_status.last_result.end_time
except Exception:
    previous_end_time = None

indexer_client.run_indexer(indexer_name)
print("Indexer started")
for attempt in range(1, 31):
    status = indexer_client.get_indexer_status(indexer_name)
    result = status.last_result
    # Only treat last_result as the current run once its end_time is newer than
    # what we observed before run_indexer was called.
    is_current_run = result and result.end_time and (previous_end_time is None or result.end_time > previous_end_time)
    if is_current_run and result.status in {"success", "transientFailure", "persistentFailure"}:
        print(f"Indexer status: {result.status}")
        if result.status != "success":
            raise RuntimeError(result.error_message or "Indexer failed")
        break
    print(f"Polling ({attempt}/30)")
    time.sleep(10)
else:
    raise TimeoutError("Indexer polling timed out")


## Step 5 - Run keyword query


In [ ]:
from IPython.display import Markdown, display

query = "argentina"
search_client = SearchClient(endpoint=search_endpoint, index_name=index_name, credential=credential)
results = list(search_client.search(search_text=query, top=5))
if not results:
    raise RuntimeError(f"No results returned for query {query!r}")


def _snippet(text: str, limit: int = 280) -> str:
    text = " ".join((text or "").split())
    return text if len(text) <= limit else text[:limit].rsplit(" ", 1)[0] + "…"


lines = [f"### Results for `{query}` ({len(results)} hits)"]
for idx, doc in enumerate(results, start=1):
    title = doc.get("title") or "(no title)"
    score = doc.get("@search.score")
    lines.append(f"**{idx}. {title}**  ·  score `{score:.3f}`" if score is not None else f"**{idx}. {title}**")
    lines.append(f"> {_snippet(doc.get('content', ''))}")
    lines.append("")
display(Markdown("\n".join(lines)))
